# Assignment 5
## Data Preprocessing

We will be using a dataframe created from *Income Dirty Data.csv*. Download the file from D2L. 

1. Import the following modules
    - `pandas`
    - `numpy`
    - `preprocessing` from `sklearn` (for bonus question)
    - `KNNImputer` from `sklearn.impute` (for bonus question)
2. Create your dataframe from the file using `pandas`

In [13]:
import pandas as pd
import numpy as np
df = pd.read_csv('./Income Dirty Data.csv')
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

3. Calculate and display the following information
    - Total number of NaN values for **each column**
    - Percentage of NaN values in the dataset 
    - Number of rows *without* any NaN values

In [14]:
def missing_values():
    print("Missing values per column:")
    null_values = df.isna().sum()
    print(null_values.to_string())

    print("\nPercentage of missing values:")
    pct_missing = null_values.sum()/(len(df) * len(df.columns))
    print(f"{pct_missing * 100:.2F}%")

    print("\nNumber of rows WITHOUT any missing values:")
    rows_with_missing = len((df[(df.ID.isna()) | (df.age.isna())]))
    print(len(df) - rows_with_missing)

missing_values()

Missing values per column:
ID              0
sex            88
age             0
income        109
tax_15_pct     93

Percentage of missing values:
5.80%

Number of rows WITHOUT any missing values:
1000


Besides missing values (NaN), the dataset contains errors. We have the following rules to check:

* All employees are adults (18+ years old)
* All employees pay 15% of their income for the tax
* All employees make money; no income should be <= 0 
---
4. Calculate and display the percentage of the data that does **NOT** violate any **one** of the rules.

In [15]:
valid_mask = (
    df["age"].notna() &
    df["income"].notna() &
    df["tax_15_pct"].notna() &
    (df["age"] >= 18) &
    (df["income"] > 0) &
    np.isclose(df["tax_15_pct"], df["income"] * 0.15)
)

valid_employees = df[valid_mask].copy()
pct_valid = (len(valid_employees)/len(df) * 100)
print(f"Percent of valid employees = %{pct_valid}")


Percent of valid employees = %59.8


Now that we have determined the number of erroneous datapoints in our set, let's work on correcting it as best we can.

5. Replace non *Female*/*Male* values in the **Sex** column with either *Female* or *Male* (e.g., Women --> Female)

In [16]:
#5
#Finding different formats for sex in the dataset
print("\nDistinct Sex Values:", df["sex"].nunique())
print(sorted(df["sex"].dropna().unique(), reverse=True)[:20])

#Replacing wrong formats with Male/Female
df["sex"] = df["sex"].replace(["Man", "Men"], "Male")
df["sex"] = df["sex"].replace(["Woman", "Women"], "Female")

#Checking Work
print("\nDistinct Sex Values After Cleaning:", df["sex"].nunique())
print(sorted(df["sex"].dropna().unique(), reverse=True)[:20])


Distinct Sex Values: 6
['Women', 'Woman', 'Men', 'Man', 'Male', 'Female']

Distinct Sex Values After Cleaning: 2
['Male', 'Female']


6. Replace non-positive **Age** values with NaN (`numpy.NaN`)
7. Replace non-positive **Income** values with NaN (`numpy.NaN`)
8. Replace non-positive **Tax (15%)** values with NaN (`numpy.NaN`)

In [17]:
def fix_neg():
    #6. Replace non-positive **Age** values with NaN
    df.loc[df['age'] <= 0, 'age'] = np.nan
    #7. Replace non-positive **Income** values with NaN
    df.loc[df['income'] <= 0, 'income'] = np.nan
    #8. Replace non-positive **Tax (15%)** values with NaN 
    df.loc[df['tax_15_pct'] <= 0, 'tax_15_pct'] = np.nan
fix_neg()
df.head(15)

,ID,sex,age,income,tax_15_pct
0,1,Female,21.0,147168.0,22075.20
1,2,Female,29.0,119595.0,17939.25
2,3,Female,56.0,87770.0,13165.50
3,4,NaN,21.0,54259.0,8138.85
4,5,Male,28.0,NaN,160230.00
5,6,Female,NaN,128326.0,19248.90
6,7,Female,NaN,NaN,NaN
7,8,Female,24.0,NaN,11820.60
8,9,Female,38.0,149473.0,22420.95
9,10,Male,48.0,113663.0,1136630.00


The following question is a bonus (+10) question, but I'd encourage you to give it a try!

9. Use machine learning (`KNNImputer`) to impute all missing values (replaces NaN values with the most predicted values)
    - Will need to use a scaler and convert the values in the **Sex** column to a numeric value for algorithm to work properly
    - Show some of the data prior to imputing, and after imputing

In [18]:
# Data prior to imputing
print("\n--------------BEFORE IMPUTING --------------")
missing_values()
df.head(10)

#Using machine learning to fill missing values
df["sex"] = df["sex"].map({"Male": 0, "Female": 1})
print(df["sex"].value_counts(dropna=False))
df.head()
columns = ["sex", "age", "income", "tax_15_pct"]
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[columns])
imputer = KNNImputer(n_neighbors=5)
imputed_data = imputer.fit_transform(scaled_data)
imputed_data = scaler.inverse_transform(imputed_data)
df[columns] = imputed_data

# Data after imputing
print("\n--------------AFTER IMPUTING --------------")
missing_values()
print(df.head(10))



--------------BEFORE IMPUTING --------------
Missing values per column:
ID              0
sex            88
age            98
income        153
tax_15_pct    103

Percentage of missing values:
8.84%

Number of rows WITHOUT any missing values:
902
sex
0.0    457
1.0    455
NaN     88
Name: count, dtype: int64

--------------AFTER IMPUTING --------------
Missing values per column:
ID            0
sex           0
age           0
income        0
tax_15_pct    0

Percentage of missing values:
0.00%

Number of rows WITHOUT any missing values:
1000
   ID  sex   age    income  tax_15_pct
0   1  1.0  21.0  147168.0    22075.20
1   2  1.0  29.0  119595.0    17939.25
2   3  1.0  56.0   87770.0    13165.50
3   4  0.2  21.0   54259.0     8138.85
4   5  0.0  28.0   87077.0   160230.00
5   6  1.0  36.2  128326.0    19248.90
6   7  1.0  40.8   76111.4    14991.18
7   8  1.0  24.0  110944.4    11820.60
8   9  1.0  38.0  149473.0    22420.95
9  10  0.0  48.0  113663.0  1136630.00


10. In the empty `Markdown` cell below, explain why it is important to clean a dataset before calculating analytics about the data

It is important to clean a dataset before calculating analytics because dirty data can lead to an innacurate analysis, as missing or invalid data can skew results. 

10. Submit this `Jupyter` file to D2L, renamed as **Last_First_Assignment4.ipynb** 
    - Replace '**Last**' and '**First**' with your name